# COMP3610 Assignment 2

- **Name:** Sonali Maharaj
- **Student ID:** 816034459
- **Course:** COMP3610  
- **Assignment:** Assignment 2

---

## Data Ingestion

In [1]:
from pathlib import Path
import requests

# Create raw data directory
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

FILES = [
    (
        "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
        RAW_DIR / "yellow_tripdata_2024-01.parquet",
    ),
    (
        "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
        RAW_DIR / "taxi_zone_lookup.csv",
    ),
]

def download_file(url, output_path, chunk_size=1024 * 1024):
    print(f"\nDownloading: {url}")

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()

        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Download failed: {output_path}")

    print(f"Saved to: {output_path}")

# Download files if they do not exist
for url, path in FILES:
    if path.exists() and path.stat().st_size > 0:
        print(f"File already exists, skipping: {path}")
    else:
        download_file(url, path)

File already exists, skipping: data\raw\yellow_tripdata_2024-01.parquet
File already exists, skipping: data\raw\taxi_zone_lookup.csv


## Data Validation

In [2]:
import polars as pl
from pathlib import Path

PARQUET_PATH = Path("data/raw/yellow_tripdata_2024-01.parquet")

if not PARQUET_PATH.exists():
    raise FileNotFoundError("Run the data ingestion step first.")

lf = pl.scan_parquet(str(PARQUET_PATH))

schema = lf.schema
actual_columns = list(schema.keys())

EXPECTED_COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type",
]

DATETIME_COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

for col in DATETIME_COLUMNS:
    if schema[col] != pl.Datetime:
        raise TypeError(f"{col} is not datetime type")

row_count = lf.select(pl.len()).collect().item()

print("Validation Passed")
print(f"Rows: {row_count:,}")
print(f"Columns: {len(actual_columns)}")

Validation Passed
Rows: 2,964,624
Columns: 19


## Data Cleaning

In [3]:
import polars as pl

def count_rows(lazyframe):
    return lazyframe.select(pl.len()).collect().item()

lf = pl.scan_parquet("data/raw/yellow_tripdata_2024-01.parquet")

start_rows = count_rows(lf)
print(f"Starting rows: {start_rows:,}")

CRITICAL_COLS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
]

# Remove rows with nulls
lf1 = lf.drop_nulls(CRITICAL_COLS)

# Remove invalid trips
lf2 = lf1.filter(
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") >= 0) &
    (pl.col("fare_amount") <= 500)
)

# Remove dropoff before pickup
lf3 = lf2.filter(
    pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime")
)

# Keep only credit card payments
lf4 = lf3.filter(pl.col("payment_type") == 1)

df_clean = lf4.collect()

print("Cleaning complete")
print("Final rows:", df_clean.height)

Starting rows: 2,964,624
Cleaning complete
Final rows: 2298412


## Feature Engineering

In [4]:
import polars as pl

if "df_clean" not in globals():
    raise RuntimeError("Run data cleaning first.")

duration_minutes_expr = (
    (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60
)

df_features = df_clean.with_columns([

    # Trip duration
    duration_minutes_expr.alias("trip_duration_minutes"),

    # Trip speed
    pl.when(duration_minutes_expr > 0)
      .then(pl.col("trip_distance") / (duration_minutes_expr / 60))
      .otherwise(0)
      .alias("trip_speed_mph"),

    # Pickup hour
    pl.col("tpep_pickup_datetime")
      .dt.hour()
      .alias("pickup_hour"),

    # Pickup day of week
    pl.col("tpep_pickup_datetime")
      .dt.strftime("%A")
      .alias("pickup_day_of_week"),

    # Tip percentage
    (pl.col("tip_amount") / pl.col("fare_amount"))
      .alias("tip_pct"),

    # High tip classification target
    (pl.col("tip_amount") / pl.col("fare_amount") >= 0.20)
      .cast(pl.Int8)
      .alias("high_tip"),
])

print("Feature engineering complete")
df_features.head()

Feature engineering complete


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_day_of_week,tip_pct,high_tip
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,str,f64,i8
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,6.6,16.363636,0,"""Monday""",0.375,1
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,17.916667,15.739535,0,"""Monday""",0.128755,0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,8.3,10.120482,0,"""Monday""",0.2,1
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,6.1,7.868852,0,"""Monday""",0.405063,1
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,"""N""",148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,32.383333,8.708183,0,"""Monday""",0.233108,1


## Saving Processed Dataset

In [5]:
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

df_features.write_parquet("data/processed/taxi_cleaned_features.parquet")

print("Saved to data/processed/taxi_cleaned_features.parquet")

Saved to data/processed/taxi_cleaned_features.parquet


## Converting to Pandas

In [6]:
import numpy as np
import pandas as pd

df = df_features.to_pandas()
zones = pd.read_csv("data/raw/taxi_zone_lookup.csv")

print("Dataset ready for modeling:", df.shape)
df.head()

Dataset ready for modeling: (2298412, 25)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_day_of_week,tip_pct,high_tip
0,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,...,1.0,18.75,2.5,0.0,6.600000,16.363636,0,Monday,0.375000,1
1,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,...,1.0,31.30,2.5,0.0,17.916667,15.739535,0,Monday,0.128755,0
2,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,...,1.0,17.00,2.5,0.0,8.300000,10.120482,0,Monday,0.200000,1
3,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,...,1.0,16.10,2.5,0.0,6.100000,7.868852,0,Monday,0.405063,1
4,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,...,1.0,41.50,2.5,0.0,32.383333,8.708183,0,Monday,0.233108,1


# Part 1: Data Preprocessing & Feature Engineering 

- In this section, the cleaned NYC taxi dataset from Assignment 1 is prepared for machine learning. New temporal, trip, and fare-based features are created to capture when trips occur, their characteristics, and cost structure. Pickup and dropoff boroughs are obtained using the taxi zone lookup table and encoded for use in machine learning models.

- Two target variables are created: `tip_amount` for the regression task and `high_tip`, a binary variable indicating whether the tip exceeds 20% of the fare amount for the classification task.

- Finally, the dataset is split into training (70%), validation (15%), and test (15%) sets using stratified sampling based on `high_tip`. Numeric features are scaled using `StandardScaler`, fitted only on the training data to prevent data leakage, and a summary of the final modeling features is provided.

## 1. Feature Engineering

### Creating the required engineered features

In [7]:
# Ensuring datetime columns are proper datetime types
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])


# a) Temporal features

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek   # Monday=0
df["is_weekend"] = df["pickup_day_of_week"].isin([5, 6])


# b) Trip features

df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Avoid division by zero for speed calculation
df["trip_speed_mph"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["trip_distance"] / (df["trip_duration_minutes"] / 60),
    np.nan
)

df["log_trip_distance"] = np.log1p(df["trip_distance"])


# c) Fare features

df["fare_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["fare_amount"] / df["trip_distance"],
    np.nan
)

df["fare_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["fare_amount"] / df["trip_duration_minutes"],
    np.nan
)

df[[
    "pickup_hour",
    "pickup_day_of_week",
    "is_weekend",
    "trip_duration_minutes",
    "trip_speed_mph",
    "log_trip_distance",
    "fare_per_mile",
    "fare_per_minute"
]].head()

,pickup_hour,pickup_day_of_week,is_weekend,trip_duration_minutes,trip_speed_mph,log_trip_distance,fare_per_mile,fare_per_minute
0,0,0,False,6.600000,16.363636,1.029619,5.555556,1.515152
1,0,0,False,17.916667,15.739535,1.740466,4.957447,1.300465
2,0,0,False,8.300000,10.120482,0.875469,7.142857,1.204819
3,0,0,False,6.100000,7.868852,0.587787,9.875000,1.295082
4,0,0,False,32.383333,8.708183,1.740466,6.297872,0.914050


### Adding pickup and dropoff borough features from the lookup table

- The taxi zone lookup table was merged with the dataset to obtain the pickup and dropoff borough for each trip using the location IDs. These borough features will later be encoded for use in machine learning models.

In [8]:
# d) Zone features

# Keeping only the lookup fields that are necessary
zone_lookup = zones[["LocationID", "Borough"]].copy()

# Merge pickup borough
pickup_lookup = zone_lookup.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "pickup_borough"
})

df = df.merge(pickup_lookup, on="PULocationID", how="left")

# Merge dropoff borough
dropoff_lookup = zone_lookup.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough"
})

df = df.merge(dropoff_lookup, on="DOLocationID", how="left")

df[["PULocationID", "pickup_borough", "DOLocationID", "dropoff_borough"]].head()

,PULocationID,pickup_borough,DOLocationID,dropoff_borough
0,140,Manhattan,236,Manhattan
1,236,Manhattan,79,Manhattan
2,79,Manhattan,211,Manhattan
3,211,Manhattan,148,Manhattan
4,148,Manhattan,141,Manhattan


In [9]:
# Data quality checks
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print(df[[
    "trip_speed_mph",
    "fare_per_minute",
    "pickup_borough",
    "dropoff_borough"
]].isnull().sum())

trip_speed_mph       33
fare_per_minute      33
pickup_borough      311
dropoff_borough    7850
dtype: int64


### Cleaning the data

In [10]:
df.dropna(subset=[
    "trip_speed_mph",
    "fare_per_minute",
    "pickup_borough",
    "dropoff_borough"
], inplace=True)

Rows containing missing values in engineered features (trip speed, fare per minute, or borough information) were removed to ensure the dataset is suitable for model training.

### One-Hot Encoding Borough Features

The pickup and dropoff borough columns are categorical variables. These features are converted into numerical format using one-hot encoding so they can be used by machine learning models.

In [11]:
df = pd.get_dummies(
    df,
    columns=["pickup_borough", "dropoff_borough"],
    prefix=["pickup", "dropoff"]
)

In [12]:
# Convert store_and_fwd_flag from categorical (Y/N) to numeric (1/0)
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].map({"Y": 1, "N": 0})

Mapping store_and_fwd_flag to numbers as the models can only train on numeric values

## 2. Target Variable Creation 

- Two target variables are defined for the modeling tasks. `tip_amount` is used as the continuous target for the regression problem. A second binary target, `high_tip`, is created to indicate whether the tip exceeds 20% of the fare amount, which will be used for the classification task.

In [13]:
# Regression target 
y_reg = df["tip_amount"]

# Classification target, tip greater than 20% of fare
df["high_tip"] = (df["tip_amount"] > 0.20 * df["fare_amount"]).astype(int)

# Preview the new target column
df[["tip_amount", "fare_amount", "high_tip"]].head()

,tip_amount,fare_amount,high_tip
0,3.75,10.0,1
1,3.00,23.3,0
2,2.00,10.0,0
3,3.20,7.9,1
4,6.90,29.6,1


## 3. Data Splitting & Scaling

- The dataset is split into training (70%), validation (15%), and test (15%) sets. Stratified sampling is used based on the `high_tip` classification target to maintain the same class distribution across splits. Numeric features are scaled using `StandardScaler`, which is fitted only on the training data to prevent data leakage. Finally, the number of samples and class distribution for each split are reported, along with a summary of the modeling features.

### Selecting features and targets

In [14]:
from sklearn.model_selection import train_test_split

# Targets
y_reg = df["tip_amount"]
y_clf = df["high_tip"]

# Features - exclude targets
X = df.drop(columns=["tip_amount", "high_tip"])

In [15]:
# Columns to exclude from features including targets and raw datetime
excluded_features = [
    "tip_amount",              
    "high_tip",               
    "tpep_pickup_datetime",    
    "tpep_dropoff_datetime"    
]
X = df.drop(columns=excluded_features)

# Exclude features that still contain NaN values for models like LinearRegression
nan_excluded_features = X.columns[X.isna().any()].tolist()
if nan_excluded_features:
    X = X.drop(columns=nan_excluded_features)
    print("Excluded NaN feature columns:", nan_excluded_features)

y_reg = df["tip_amount"]
y_clf = df["high_tip"]

Excluded NaN feature columns: ['tip_pct']


Raw datetime columns (`tpep_pickup_datetime` and `tpep_dropoff_datetime`) were excluded from modeling because temporal information was already captured through engineered features such as `pickup_hour`, `pickup_day_of_week`, and `trip_duration_minutes`. The target variables `tip_amount` and `high_tip` were also excluded from the feature set to prevent data leakage. Any additional feature columns that contained NaN values were excluded as well so models like Linear Regression can train without missing-value errors. In this run, `tip_pct` was excluded for this reason.

### Train / validation / test split

In [16]:
# First split - 70% train, 30% temp

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_clf,
    test_size=0.30,
    stratify=y_clf,
    random_state=42
)

# Second split - 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

### Documenting number of samples

In [17]:
print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 1603286
Validation size: 343561
Test size: 343562


### Checking class distribution

In [18]:
print("Train high_tip distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation high_tip distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest high_tip distribution:")
print(y_test.value_counts(normalize=True))

Train high_tip distribution:
high_tip
1    0.760076
0    0.239924
Name: proportion, dtype: float64

Validation high_tip distribution:
high_tip
1    0.760078
0    0.239922
Name: proportion, dtype: float64

Test high_tip distribution:
high_tip
1    0.760075
0    0.239925
Name: proportion, dtype: float64


The dataset was split into training (70%), validation (15%), and test (15%) sets using stratified sampling based on the `high_tip` target. The class distribution remains consistent across all splits, with approximately 76% high-tip trips and 24% normal-tip trips, confirming that stratification was successful.

In [19]:
# Remove datetime columns since models require numeric inputs
datetime_cols = X_train.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns

X_train = X_train.drop(columns=datetime_cols)
X_val = X_val.drop(columns=datetime_cols)
X_test = X_test.drop(columns=datetime_cols)

The original pickup and dropoff datetime columns were excluded from modeling because machine learning models require numeric input. Their useful information was already captured through engineered temporal features such as pickup hour, day of week, weekend indicator, and trip duration.

In [20]:
# Remove columns that leak information about the target
leakage_cols = ["total_amount"]

X_train = X_train.drop(columns=leakage_cols, errors="ignore")
X_val = X_val.drop(columns=leakage_cols, errors="ignore")
X_test = X_test.drop(columns=leakage_cols, errors="ignore")

### Scaling numeric features

- Numeric features are standardized using `StandardScaler` so that variables with different ranges do not disproportionately influence the machine learning models. The scaler is fitted only on the training data and then applied to the validation and test sets to prevent data leakage.

In [21]:
from sklearn.preprocessing import StandardScaler

# Select only numeric columns
numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns

scaler = StandardScaler()

# Keep dataframe structure
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Scale numeric columns only
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

### Feature summary 

- The final feature set includes engineered temporal, trip, fare, and encoded borough features. The target variables and raw datetime columns are excluded from the model inputs to prevent data leakage and because their information is already captured by engineered features.

In [22]:
# Create a feature summary table
feature_summary = pd.DataFrame({
    "feature_name": X.columns,
    "data_type": X.dtypes.astype(str).values
})

print("Number of features used for modeling:", len(X.columns))
display(feature_summary)

# Excluded features with reasons
excluded_feature_names = [
    "tip_amount",
    "high_tip",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
] + nan_excluded_features

excluded_feature_reasons = [
    "Regression target variable",
    "Classification target variable",
    "Raw datetime column; temporal information already captured by engineered features",
    "Raw datetime column; trip duration already engineered",
] + ["Excluded due to NaN values"] * len(nan_excluded_features)

excluded_summary = pd.DataFrame({
    "excluded_feature": excluded_feature_names,
    "reason": excluded_feature_reasons
})

display(excluded_summary)

Number of features used for modeling: 38


,feature_name,data_type
0,VendorID,int32
1,passenger_count,int64
2,trip_distance,float64
3,RatecodeID,int64
4,store_and_fwd_flag,int64
5,PULocationID,int32
6,DOLocationID,int32
7,payment_type,int64
8,fare_amount,float64
9,extra,float64


,excluded_feature,reason
0,tip_amount,Regression target variable
1,high_tip,Classification target variable
2,tpep_pickup_datetime,Raw datetime column; temporal information alre...
3,tpep_dropoff_datetime,Raw datetime column; trip duration already eng...
4,tip_pct,Excluded due to NaN values


The table above shows the final features used for modeling and their data types. These include engineered temporal, trip, and fare features as well as one-hot encoded borough features. Target variables and raw datetime columns were excluded from the feature set to prevent data leakage.

### Observations During Preprocessing

After filtering the dataset to include only credit card transactions, the dataset contains approximately 2.29 million trips. Only a small number of rows contained missing values in engineered features such as trip speed or borough information, and these rows were removed during preprocessing.

The `high_tip` target shows a moderate class imbalance, with approximately 76% of trips classified as high-tip and 24% as normal-tip trips. Stratified sampling was used during the train/validation/test split to maintain this class distribution across all datasets.

# Part 2: Model Training & Tuning

- In this section, baseline machine learning models are trained for both regression and classification. For the regression task, Linear Regression and Random Forest Regressor are used to predict `tip_amount`. For the classification task, Logistic Regression and Random Forest Classifier are used to predict `high_tip`.

- Model performance is evaluated on the validation set. Regression models are assessed using MAE, RMSE, and R², while classification models are assessed using accuracy, precision, recall, F1-score, and AUC-ROC.

## 4. Baseline Models

### Ensuring that targets match the split rows

In [23]:
# Regression targets
y_train_reg = y_reg.loc[X_train.index]
y_val_reg = y_reg.loc[X_val.index]
y_test_reg = y_reg.loc[X_test.index]

# Classification targets
y_train_clf = y_clf.loc[X_train.index]
y_val_clf = y_clf.loc[X_val.index]
y_test_clf = y_clf.loc[X_test.index]

The regression and classification target variables were aligned with the existing train, validation, and test splits by selecting the target values corresponding to the indices of the feature datasets.

### Importing models 

In [24]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### Baseline Regression Models

- Two baseline regression models are trained to predict `tip_amount`: Linear Regression and Random Forest Regressor. Linear Regression is trained on scaled features, while Random Forest Regressor is trained on the original unscaled features.

In [ ]:
# Store results for each model

regression_results = []

# Remove any remaining NaN columns before Linear Regression
nan_cols_lr = X_train_scaled.columns[X_train_scaled.isna().any()].tolist()
if nan_cols_lr:
    X_train_scaled = X_train_scaled.drop(columns=nan_cols_lr)
    X_val_scaled = X_val_scaled.drop(columns=nan_cols_lr)
    X_train = X_train.drop(columns=nan_cols_lr, errors="ignore")
    X_val = X_val.drop(columns=nan_cols_lr, errors="ignore")
    print("Excluded NaN feature columns before regression:", nan_cols_lr)

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_reg)

y_val_pred_lr = lr.predict(X_val_scaled)

# Calculate evaluation metrics and store results
regression_results.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(y_val_reg, y_val_pred_lr),
    "RMSE": np.sqrt(mean_squared_error(y_val_reg, y_val_pred_lr)),
    "R2": r2_score(y_val_reg, y_val_pred_lr)
})

# 2. Random Forest Regressor
rf_reg = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

rf_reg.fit(X_train, y_train_reg)

# predict tip_amount on validation data
y_val_pred_rf_reg = rf_reg.predict(X_val)

# Calculate evaluation metrics and store results
regression_results.append({
    "Model": "Random Forest Regressor",
    "MAE": mean_absolute_error(y_val_reg, y_val_pred_rf_reg),
    "RMSE": np.sqrt(mean_squared_error(y_val_reg, y_val_pred_rf_reg)),
    "R2": r2_score(y_val_reg, y_val_pred_rf_reg)
})

# Convert results into a dataframe 
regression_results_df = pd.DataFrame(regression_results)

# Display regression model performance
regression_results_df

The regression models show moderate performance. Linear Regression performed slightly better than the Random Forest Regressor with an R² of 0.63 compared to 0.61. Both models have similar error levels, with average prediction errors of about $1.20.

### Baseline Classification Models

- Two baseline classification models are trained to predict `high_tip`: Logistic Regression and Random Forest Classifier. Logistic Regression is trained on scaled features, while Random Forest Classifier is trained on the original unscaled features.

In [ ]:
# Store classification model results
classification_results = []

# 1. Logistic Regression
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# train model using scaled training data
log_reg.fit(X_train_scaled, y_train_clf)

# predicted class labels for validation set
y_val_pred_log = log_reg.predict(X_val_scaled)

y_val_prob_log = log_reg.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics and store results
classification_results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val_clf, y_val_pred_log),
    "Precision": precision_score(y_val_clf, y_val_pred_log),
    "Recall": recall_score(y_val_clf, y_val_pred_log),
    "F1": f1_score(y_val_clf, y_val_pred_log),
    "AUC-ROC": roc_auc_score(y_val_clf, y_val_prob_log)
})

# 2. Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train_clf)

# predicted class labels
y_val_pred_rf_clf = rf_clf.predict(X_val)

y_val_prob_rf_clf = rf_clf.predict_proba(X_val)[:, 1]

# Calculate evaluation metrics and store results
classification_results.append({
    "Model": "Random Forest Classifier",
    "Accuracy": accuracy_score(y_val_clf, y_val_pred_rf_clf),
    "Precision": precision_score(y_val_clf, y_val_pred_rf_clf),
    "Recall": recall_score(y_val_clf, y_val_pred_rf_clf),
    "F1": f1_score(y_val_clf, y_val_pred_rf_clf),
    "AUC-ROC": roc_auc_score(y_val_clf, y_val_prob_rf_clf)
})

# Convert results to a dataframe
classification_results_df = pd.DataFrame(classification_results)

# Display classification model performance
classification_results_df

The classification models achieved moderate performance with accuracies around 77%. Both models identify most high-tip trips but have relatively low AUC-ROC scores. Logistic Regression performed slightly better than Random Forest.

### Summary 

In [ ]:
print("Regression Model Validation Results")
display(regression_results_df)

print("Classification Model Validation Results")
display(classification_results_df)

### Model Performance on the Validation Set

- The baseline models were evaluated on the validation set using the required metrics.

- For the regression task (`tip_amount`), Linear Regression achieved an MAE of 1.18, RMSE of 2.28, and R² of 0.63. The Random Forest Regressor produced similar results with an MAE of 1.21, RMSE of 2.33, and R² of 0.61.

- For the classification task (`high_tip`), Logistic Regression achieved an accuracy of 0.77, precision of 0.77, recall of 0.999, F1-score of 0.87, and AUC-ROC of 0.61. The Random Forest Classifier had similar performance with an accuracy of 0.76, precision of 0.77, recall of 0.98, F1-score of 0.86, and AUC-ROC of 0.59.

- Overall, Logistic Regression performed slightly better for the classification task, while Linear Regression performed slightly better for the regression task.

## 5. Hyperparameter Tuning

- For hyperparameter tuning, `RandomizedSearchCV` will be used on the Random Forest Regressor. Although Linear Regression performed slightly better in the baseline results, it has limited hyperparameters to tune. Random Forest is more suitable for this step because it has several important hyperparameters that can be optimized.

- To reduce computation time, we will use a sample of the training set and perform 5-fold cross-validation. 

### Sampling the training set

In [ ]:
# Take a sample of the training data for faster tuning
sample_size = min(200000, len(X_train))

X_train_sample = X_train.sample(n=sample_size, random_state=42)
y_train_reg_sample = y_train_reg.loc[X_train_sample.index]

print("Sample size used for tuning:", len(X_train_sample))

### Running RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Define the hyperparameter search space
param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

# Initialize the model
rf_reg_tune = RandomForestRegressor(random_state=42, n_jobs=-1)

# Perform randomized search with 5-fold cross-validation
random_search = RandomizedSearchCV(
    estimator=rf_reg_tune,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="r2",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_sample, y_train_reg_sample)

### Hyperparameter Tuning Results

RandomizedSearchCV with 5-fold cross-validation was used to tune the Random Forest Regressor, testing 20 hyperparameter combinations (n_iter=20), resulting in 100 model fits. The best model used max_depth=10, max_features='sqrt', and min_samples_split=5, which produced the highest cross-validation R² score. The final estimator was RandomForestRegressor(max_depth=10, max_features='sqrt', min_samples_split=5, n_jobs=-1, random_state=42).

### Showing the best parameters


In [ ]:
# Display best parameters
print("Best Parameters:", random_search.best_params_)

### Train tuned model and compare on validation set

In [ ]:
# Train the tuned model on the full training set
best_rf_reg = random_search.best_estimator_
best_rf_reg.fit(X_train, y_train_reg)

# Predict on the validation set
y_val_pred_tuned = best_rf_reg.predict(X_val)

# Evaluate tuned model
tuned_results = pd.DataFrame([{
    "Model": "Tuned Random Forest Regressor",
    "MAE": mean_absolute_error(y_val_reg, y_val_pred_tuned),
    "RMSE": np.sqrt(mean_squared_error(y_val_reg, y_val_pred_tuned)),
    "R2": r2_score(y_val_reg, y_val_pred_tuned)
}])

tuned_results

The tuned Random Forest Regressor achieved slightly better performance on the validation set, with lower MAE and RMSE and a higher R² compared to the baseline model.

### Compare baseline vs tuned

In [ ]:
# Compare baseline and tuned Random Forest results
comparison_df = pd.concat(
    [
        regression_results_df[regression_results_df["Model"] == "Random Forest Regressor"],
        tuned_results
    ],
    ignore_index=True
)

comparison_df

### Baseline vs Tuned Model Comparison

The tuned Random Forest Regressor showed a small improvement compared to the baseline model. The tuned model achieved lower MAE and RMSE values and a higher R² score on the validation set, indicating slightly better predictive performance.

## 6. Neural Network Model

- For the neural network model, PyTorch will be used to build a feedforward neural network for the classification task of predicting `high_tip`. The network includes two hidden layers and is trained using mini-batches with a DataLoader. `BCEWithLogitsLoss` will be used as the loss function and Adam as the optimizer, and the model will be trained for at least 20 epochs.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

### Preparing tensors

In [ ]:
# Convert scaled features and targets to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled.to_numpy(), dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled.to_numpy(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_clf.to_numpy(), dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val_clf.to_numpy(), dtype=torch.float32).view(-1, 1)

# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create DataLoaders for batch training
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

### Defining the neural network

In [ ]:
# Define a feedforward neural network with two hidden layers
class HighTipNN(nn.Module):
    def __init__(self, input_dim):
        super(HighTipNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

### Initializing the model, loss, optimizer

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model, loss function, and optimizer
input_dim = X_train_tensor.shape[1]

# Create the neural network
model = HighTipNN(input_dim).to(device)

# Define the loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### Training loop

In [ ]:
# Train the neural network and track training/validation loss
num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    
    # Looping through the batches
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Switch to eval mode
    model.eval()

    # Track validation loss
    running_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

### Plotting loss curves

In [ ]:
# Plot training and validation loss curves
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Training and Validation Loss Curves

The training and validation loss both decrease slightly during training and remain very close to each other. This suggests that the neural network is learning from the data and is not overfitting. The loss levels off after several epochs, indicating that the model has mostly finished learning.

### Evaluating on validation set

In [ ]:
# Evaluate the neural network on the validation set
model.eval()

all_probs = []
all_preds = []
all_true = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        # Convert logits to probabilities
        probs = torch.sigmoid(logits)
        # Convert probabilities to predictions
        preds = (probs >= 0.5).float()

        # Store results
        all_probs.extend(probs.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())
        all_true.extend(y_batch.numpy().flatten())

# Calculate classification metrics
nn_results = pd.DataFrame([{
    "Model": "PyTorch Neural Network",
    "Accuracy": accuracy_score(all_true, all_preds),
    "Precision": precision_score(all_true, all_preds, zero_division=0),
    "Recall": recall_score(all_true, all_preds, zero_division=0),
    "F1": f1_score(all_true, all_preds, zero_division=0),
    "AUC-ROC": roc_auc_score(all_true, all_probs)
}])

nn_results

The PyTorch neural network achieved similar performance to the Scikit-learn models, with an accuracy of 0.77 and a high recall for detecting high-tip trips.

### Comparing with Scikit-learn models

In [ ]:
# Compare neural network results with baseline classification models
classification_comparison_df = pd.concat(
    [classification_results_df, nn_results],
    ignore_index=True
)

classification_comparison_df

The tuned Random Forest Regressor performed slightly better than the baseline model on the validation set. It achieved lower MAE and RMSE and a higher R² score, indicating improved predictive performance after hyperparameter tuning.

# Part 3: Model Evaluation & Interpretation

## 7. Comprehensive Evaluation

- In this section, we will evaluate all trained models on the held-out test set. 

- For the classification task, we will create a summary table of all models, plot ROC curves on the same figure, and generate a confusion matrix for the best-performing model.

- For the regression task, we will create a summary table of all models, plot predicted vs actual tip amounts for the best model, and perform residual analysis using a residual distribution plot and a residuals vs predicted values plot.

### Regression summary table 

In [ ]:
# Evaluate regression models on the test set
regression_test_results = []

# Linear Regression
y_test_pred_lr = lr.predict(X_test_scaled)

regression_test_results.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(y_test_reg, y_test_pred_lr),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, y_test_pred_lr)),
    "R2": r2_score(y_test_reg, y_test_pred_lr)
})

# Random Forest Regressor
y_test_pred_rf_reg = rf_reg.predict(X_test)

regression_test_results.append({
    "Model": "Random Forest Regressor",
    "MAE": mean_absolute_error(y_test_reg, y_test_pred_rf_reg),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, y_test_pred_rf_reg)),
    "R2": r2_score(y_test_reg, y_test_pred_rf_reg)
})

# Tuned Random Forest Regressor
y_test_pred_tuned_rf = best_rf_reg.predict(X_test)

regression_test_results.append({
    "Model": "Tuned Random Forest Regressor",
    "MAE": mean_absolute_error(y_test_reg, y_test_pred_tuned_rf),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, y_test_pred_tuned_rf)),
    "R2": r2_score(y_test_reg, y_test_pred_tuned_rf)
})

regression_summary_df = pd.DataFrame(regression_test_results)
regression_summary_df

The regression results show that the Tuned Random Forest Regressor performed the best on the test set, achieving the lowest MAE and RMSE and the highest R² score. This indicates that hyperparameter tuning slightly improved the model's ability to predict tip amounts compared to the baseline models.

### Classification summary table

In [ ]:
# Evaluate classification models on the test set
classification_test_results = []

# Logistic Regression
y_test_pred_log = log_reg.predict(X_test_scaled)
y_test_prob_log = log_reg.predict_proba(X_test_scaled)[:,1]

classification_test_results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test_clf, y_test_pred_log),
    "Precision": precision_score(y_test_clf, y_test_pred_log, zero_division=0),
    "Recall": recall_score(y_test_clf, y_test_pred_log, zero_division=0),
    "F1": f1_score(y_test_clf, y_test_pred_log, zero_division=0),
    "AUC-ROC": roc_auc_score(y_test_clf, y_test_prob_log)
})

# Random Forest Classifier
y_test_pred_rf_clf = rf_clf.predict(X_test)
y_test_prob_rf_clf = rf_clf.predict_proba(X_test)[:,1]

classification_test_results.append({
    "Model": "Random Forest Classifier",
    "Accuracy": accuracy_score(y_test_clf, y_test_pred_rf_clf),
    "Precision": precision_score(y_test_clf, y_test_pred_rf_clf, zero_division=0),
    "Recall": recall_score(y_test_clf, y_test_pred_rf_clf, zero_division=0),
    "F1": f1_score(y_test_clf, y_test_pred_rf_clf, zero_division=0),
    "AUC-ROC": roc_auc_score(y_test_clf, y_test_prob_rf_clf)
})

### Evaluate PyTorch Neural Network

In [ ]:
classification_test_results.append({
    "Model": "PyTorch Neural Network",
    "Accuracy": accuracy_score(all_true, all_preds),
    "Precision": precision_score(all_true, all_preds, zero_division=0),
    "Recall": recall_score(all_true, all_preds, zero_division=0),
    "F1": f1_score(all_true, all_preds, zero_division=0),
    "AUC-ROC": roc_auc_score(all_true, all_probs)
})

### Create the summary table

In [ ]:
classification_summary_df = pd.DataFrame(classification_test_results)
classification_summary_df

The classification models achieved similar performance on the test set, with accuracies around 0.77. The PyTorch Neural Network produced a slightly higher AUC-ROC compared to the other models, indicating a small improvement in distinguishing between high-tip and low-tip trips.

### ROC Curves and Confusion Matrix

To further evaluate the classification models, ROC curves are plotted on the same figure to compare their ability to distinguish between high-tip and low-tip trips. A confusion matrix is also plotted for the best-performing classification model based on AUC-ROC.

### ROC curves for all classification models

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# Compute ROC curve points for each classification model
fpr_log, tpr_log, _ = roc_curve(y_test_clf, y_test_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test_clf, y_test_prob_rf_clf)
fpr_nn, tpr_nn, _ = roc_curve(all_true, all_probs)

# Plot ROC curves on the same figure
plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest Classifier")
plt.plot(fpr_nn, tpr_nn, label="PyTorch Neural Network")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Classification Models")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

The ROC curves compare the classification performance of all models on the test set. All models perform similarly, but the PyTorch Neural Network achieves the slightly highest AUC-ROC, indicating a small improvement in distinguishing between high-tip and low-tip trips.

### Confusion matrix for the best model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Create confusion matrix for the best classification model
cm = confusion_matrix(all_true, all_preds)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - PyTorch Neural Network")
plt.show()

The confusion matrix shows the predictions of the PyTorch Neural Network on the test set. The model correctly identifies most high-tip trips, resulting in very high recall, but it incorrectly classifies many low-tip trips as high-tip due to class imbalance in the dataset.

### Predicted vs Actual Tip Amounts

A scatter plot of predicted versus actual tip amounts is used to evaluate the performance of the best regression model. 

In [ ]:
import matplotlib.pyplot as plt

# Predict tip amounts on the test set using the best regression model
y_test_pred_best = best_rf_reg.predict(X_test)

# Create scatter plot
plt.figure(figsize=(8,6))
plt.scatter(y_test_reg, y_test_pred_best, alpha=0.3)

# Ideal prediction line
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         color='red', linestyle='--')

plt.xlabel("Actual Tip Amount")
plt.ylabel("Predicted Tip Amount")
plt.title("Predicted vs Actual Tip Amounts (Tuned Random Forest)")
plt.grid(True, alpha=0.3)

plt.show()

The scatter plot compares the predicted and actual tip amounts for the tuned Random Forest regression model. Many predictions follow the diagonal line, indicating reasonable accuracy for typical tip values. However, larger tip amounts are harder for the model to predict accurately, resulting in some deviations from the ideal line.

### Residual Analysis

Residual analysis helps evaluate how well the regression model fits the data. Residuals are the differences between the actual and predicted tip amounts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals
residuals = y_test_reg - y_test_pred_best

# Create figure with two plots
plt.figure(figsize=(12,5))

# Residual distribution
plt.subplot(1,2,1)
sns.histplot(residuals, bins=50, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Frequency")

# Residuals vs predicted
plt.subplot(1,2,2)
plt.scatter(y_test_pred_best, residuals, alpha=0.3)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals vs Predicted Values")
plt.xlabel("Predicted Tip Amount")
plt.ylabel("Residual")

plt.tight_layout()
plt.show()

The residual distribution shows that most prediction errors are centered around zero, indicating that the model generally predicts tip amounts accurately. The residuals vs predicted plot does not show a strong pattern, suggesting the errors are mostly random, although a few large outliers are present for extreme tip values.

## 8. Feature Importance

To better understand which features influence model predictions, feature importance is analyzed using the Random Forest model and regression coefficients from the Linear/Logistic Regression models.

### Random Forest Feature Importance

In [ ]:
# Extract feature importances from the tuned Random Forest model
feature_importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf_reg.feature_importances_
})

# Sort features by importance
feature_importances = feature_importances.sort_values(by="Importance", ascending=False)

# Plot feature importance
plt.figure(figsize=(10,6))
plt.barh(feature_importances["Feature"][:15], feature_importances["Importance"][:15])
plt.gca().invert_yaxis()

plt.title("Top Feature Importances - Random Forest Regressor")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.show()

The Random Forest feature importance plot shows that trip distance and fare amount are the most influential features for predicting tip amount. This suggests that longer and more expensive trips tend to result in higher tips. Other features such as trip duration, fare per mile, and toll amounts also contribute to the model’s predictions but have smaller impacts.

### Linear / Logistic Regression Coefficients

In [ ]:
# Extract coefficients from logistic regression
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": log_reg.coef_[0]
})

# Sort by absolute importance
coefficients["Abs_Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values(by="Abs_Coefficient", ascending=False)

coefficients.head(10)

Features such as trip distance and toll amounts positively influence the probability of higher tips, while variables like RatecodeID and log-transformed trip distance show a negative relationship in the model.

### Optional: Use SHAP values to explain individual predictions for 3 sample trips

SHAP values are used to explain how individual features contribute to a model's prediction for specific trips. Positive SHAP values push the prediction higher, while negative SHAP values push it lower.

SHAP is used only for the optional feature explanation section. 
If SHAP is not installed, the following cell installs it.

In [ ]:
%pip install shap

### Compute SHAP values for 3 sample trips

In [ ]:
import shap
import numpy as np


# Take 3 sample trips from the test set
X_sample = X_test.sample(3, random_state=42)

# Select the top 8 most important features from the Random Forest model
top_features = feature_importances["Feature"].head(8).tolist()

# Create SHAP explainer for the tuned Random Forest model
explainer = shap.TreeExplainer(best_rf_reg)

# Compute SHAP values
shap_values = explainer.shap_values(X_sample)

# Show the sampled trips
X_sample[top_features]

The table shows three sample trips using the most important features from the Random Forest model. Features such as trip distance, fare amount, and trip duration play a major role in predicting the tip amount, as longer or more expensive trips generally result in higher tips.

## 9. Written Analysis 

### a) Model Performance

For the regression task, the Tuned Random Forest Regressor performed the best, achieving the lowest MAE and RMSE and the highest R² score on the test set. This suggests that the Random Forest model was better able to capture non-linear relationships and interactions between features compared to Linear Regression. For the classification task, all models performed similarly, with accuracies around 0.77. The PyTorch Neural Network achieved the highest AUC-ROC, indicating a slightly better ability to distinguish between high-tip and low-tip trips, although the performance difference was small because the dataset consists of structured tabular data where traditional machine learning models often perform well.

### b) Predictive Features

The feature importance analysis showed that variables such as trip distance, fare amount, trip duration, and fare per mile were among the most influential predictors of tip amount. This aligns with my intuition, as longer or more expensive trips typically result in higher tips. Location features and additional charges such as tolls or airport fees may also influence tipping behavior depending on the type of trip.

### c) Model Limitations

There are several limitations to the models used in this analysis. First, some engineered features may introduce indirect information leakage if they are closely related to the target variable, which required removing certain features to prevent this issue. Additionally, the dataset may contain biases related to geographic locations or payment types that influence tipping behavior. The dataset also includes extreme outliers in tip amounts, which can make regression predictions more difficult and reduce model accuracy.

### d) Potential Improvements

Given more time or additional data, several improvements could be explored. More advanced feature engineering could be applied, such as incorporating weather conditions, time-of-day patterns, or traffic conditions. Additional hyperparameter tuning and cross-validation could further improve model performance. More robust handling of outliers or alternative regression models could also improve prediction accuracy.

### e) Neural Network vs Traditional Models

In this problem, the neural network performed similarly to the traditional machine learning models. This is likely because the dataset consists mostly of structured tabular data, where models like Random Forest and Logistic Regression often perform very well. Neural networks tend to show greater advantages when working with very large datasets or more complex data types such as images or text.

# AI Tools Used

AI assistance (ChatGPT) was used in the following ways during this assignment:

- Helping clarify assignment instructions and requirements.
- Providing general guidance on how to implement certain code steps and machine learning workflows.
- Debugging coding issues, such as handling datetime columns and encoding the `store_and_fwd_flag` feature.
- Validating whether outputs from different code cells (model results, evaluation metrics, and plots) appeared reasonable.
- Giving suggestions when multiple approaches were possible (e.g., choosing between RandomizedSearchCV and GridSearchCV).
- Assisting with understanding feature importance and exploring the optional SHAP explanation section.
- To generate a readme file, .gitignore and requirements.txt

